In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm
from itertools import combinations


In [2]:
cd ..

/ewsc/exxact04/vshir/tcga


In [3]:
labelled_data = pd.read_pickle('pkl/GI_cluster50_labelled_all_mar28.pkl') #cancer type, TCGA ID, label, X,Y coordinate 
pairwise_slide_df = pd.read_pickle('pkl/cluster_50_pairwise_slide_04_06_26.pkl') #cancer type, TCGA ID, cluster 1, cluster 2, contact count, frequency 
pairwise_summary_df = pd.read_pickle('pkl/cluster_50_pairwise_summary_04_06_26.pkl') #summary of CN1 CN2 frequencies for each cancer type 

In [4]:
labelled_data.index.set_names(['cancer_type', 'tcga_id', 'number'], inplace=True)
labelled_data

label  \
cancer_type tcga_id                                            number         
paad        TCGA-HZ-A9TJ-01Z-00-DX1.CB2953B6-9601-4300-BC89... 0          0   
                                                               1          0   
                                                               2         46   
                                                               3         46   
                                                               4         46   
...                                                                     ...   
read        TCGA-F5-6861-01Z-00-DX1.011B771B-F52E-412E-9352... 36609     13   
                                                               36610     29   
                                                               36611     29   
                                                               36612     13   
                                                               36613     13   

                                                                           X  \
cancer_type tcga_id                                            number          
paad        TCGA-HZ-A9TJ-01Z-00-DX1.CB2953B6-9601-4300-BC89... 0        1792   
                                                               1        1792   
                                                               2        2048   
                                                               3        2048   
                                                               4        2048   
...                                                                      ...   
read        TCGA-F5-6861-01Z-00-DX1.011B771B-F52E-412E-9352... 36609   94208   
                                                               36610   94208   
                                                               36611   94208   
                                                               36612   94208   
                                                               36613   94208   

                                                                           Y  
cancer_type tcga_id                                            number         
paad        TCGA-HZ-A9TJ-01Z-00-DX1.CB2953B6-9601-4300-BC89... 0       11264  
                                                               1       11520  
                                                               2        9984  
                                                               3       10240  
                                                               4       10496  
...                                                                      ...  
read        TCGA-F5-6861-01Z-00-DX1.011B771B-F52E-412E-9352... 36609   20736  
                                                               36610   20992  
                                                               36611   21248  
                                                               36612   21504  
                                                               36613   21760  

[85078334 rows x 3 columns]

In [5]:
pairwise_slide_df.sort_values(['cancer_type', 'cluster 1', 'cluster 2']).head()

,cancer_type,tcga_id,cluster 1,cluster 2,contact count,total cluster 1,total cluster 2,total_c1_interactions,frequency
4196,coad,TCGA-D5-6536-01Z-00-DX1.a4528e3f-770e-4271-994...,0,1,2,5916,18,45154,0.000044
4707,coad,TCGA-DM-A28G-01Z-00-DX1.5e8602bd-31e1-4813-821...,0,1,1,988,574,7673,0.000130
6348,coad,TCGA-DM-A1DA-01Z-00-DX1.00001FEF-3B63-4C6F-952...,0,1,1,112,60,870,0.001149
7362,coad,TCGA-D5-6537-01Z-00-DX1.f81ccf91-7ce6-4ccc-827...,0,1,8,2407,595,18198,0.000440
10247,coad,TCGA-G4-6293-01Z-00-DX1.62ed5ed9-a79a-487a-bd6...,0,1,9,15000,254,118167,0.000076


In [6]:
pairwise_summary_df.sort_values(['cancer_type', 'cluster 1', 'cluster 2']).head()

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency
0,coad,0,1,77,239810,1871860,0.000041
1,coad,0,2,3480,612543,4738758,0.000734
2,coad,0,3,58,48398,370757,0.000156
3,coad,0,4,16,1003,6726,0.002379
4,coad,0,5,554,282791,2190947,0.000253


In [7]:
label_counts = labelled_data.groupby('tcga_id')['label'].value_counts().unstack(fill_value=0)
all_labels = list(range(50))
label_counts = label_counts.reindex(columns=all_labels, fill_value=0)

In [8]:
label_counts

label,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
tcga_id,,,,,,,,,,,,,,,,,,,,,
TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01,0,0,5,7208,3423,0,0,0,0,0,...,0,0,0,0,112,0,249,12,0,0
TCGA-05-4245-01Z-00-DX1.36ff5403-d4bb-4415-b2c5-7c750d655cde,0,2,473,3786,2366,0,0,0,0,0,...,0,0,0,0,146,0,130,1,30,0
TCGA-05-4249-01Z-00-DX1.9fce0297-cc19-4c04-872c-4466ee4024db,0,0,6,13905,2330,0,5,0,0,0,...,0,0,1,0,0,0,315,1,46,0
TCGA-05-4250-01Z-00-DX1.90f67fdf-dff9-46ca-af71-0978d7c221ba,0,1,28,8944,4908,0,0,0,0,0,...,0,1,0,0,4,0,165,15,15,0
TCGA-05-4382-01Z-00-DX1.76b49a4c-dbbb-48b0-b677-6d3037e5ce88,0,7,21,8241,2318,0,0,0,0,0,...,0,1,0,0,54,0,605,66,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-Z6-A9VB-01Z-00-DX1.D35A32ED-DFEE-4070-9542-F91726BDB6A2,0,685,174,0,0,627,177,1,0,2,...,0,229,346,1,915,0,2155,37,4502,7
TCGA-Z6-AAPN-01Z-00-DX1.345F7431-CD48-40DB-89D6-A419E5BA1679,1,1533,75,0,0,337,2036,28,0,0,...,2,14805,636,6,196,1,7343,297,0,62
TCGA-ZA-A8F6-01Z-00-DX1.89510833-5BC8-4983-817E-0E9244824168,1,420,15,0,0,336,5471,464,0,2,...,549,3058,832,1125,3,0,3080,239,0,8192


In [14]:
    n_permutations = 2
    stride = 256
    num_clusters = 50
    offsets = [(stride, 0), (-stride, 0), (0, stride), (0, -stride),
               (stride, stride), (-stride, -stride), (stride, -stride), (-stride, stride)]
    cancer_types = ['coad', 'esca', 'paad', 'stad', 'read']
    
    all_pairs = []
    for ct in cancer_types:
        for c1, c2 in combinations(range(num_clusters), 2):
            all_pairs.append({'cancer_type': ct, 'cluster 1': c1, 'cluster 2': c2})
    
    null_freq_dist = pd.DataFrame(all_pairs)
    num_pairs = len(null_freq_dist)
    null_data = labelled_data.copy() #data for shuffling 
    
    # Pre-allocate a NumPy array for the frequencies
    # Rows = pairs, Columns = permutations
    freq_matrix = np.zeros((num_pairs, n_permutations))
    
    # Create a mapping for quick lookup: (cancer_type, c1, c2) -> row_index
    pair_to_idx = {tuple(x): i for i, x in enumerate(null_freq_dist.values)}
    
    # 2. Main Permutation Loop

    for i in tqdm(range(n_permutations), desc="Overall Permutation Progress"):
        slide_results = [] 
        # 3. Shuffle labels across entire dataset 
        null_data['label'] = np.random.permutation(labelled_data['label'].values)
    
        for ct in tqdm(cancer_types, desc='Cancer Type'): #can add an inner progress bar here 
            ct_data = null_data.loc[ct]
            slide_ids = ct_data.index.get_level_values('tcga_id').unique() 
            
            for sid in tqdm(slide_ids, desc='TCGA Slides'): #can add another progress bar here 
                slide_df = ct_data.loc[sid]
                coord_map = dict(zip(zip(slide_df['X'], slide_df['Y']), slide_df['label']))
                contact_matrix = np.zeros((num_clusters, num_clusters), dtype=int)
               
                # Spatial Adjacency Tally
                for (x, y), l_center in coord_map.items():
                    for dx, dy in offsets:
                        neighbor = (x + dx, y + dy)
                        if neighbor in coord_map:
                            l_neigh = coord_map[neighbor]
                            contact_matrix[l_center, l_neigh] += 1
                
                # Calculate total interactions for each cluster in this slide
                slide_totals = {c: contact_matrix[c, :].sum() for c in range(num_clusters)} 
                
                for c1, c2 in combinations(range(num_clusters), 2): 
                    contact_count = contact_matrix[c1, c2]
                    if contact_count > 0:
                        c1_total_neighbors = slide_totals[c1]
                        slide_results.append({ 
                            'cancer_type': ct,
                            'tcga_id': sid,
                            'cluster 1': c1,
                            'cluster 2': c2,
                            'contact count': contact_count,
                            'total_c1_interactions': c1_total_neighbors,
                            'frequency': contact_count / c1_total_neighbors if c1_total_neighbors > 0 else 0
                        })
        final_contact_df = pd.DataFrame(slide_results) 

        aggregated_df = final_contact_df.groupby(['cancer_type', 'cluster 1', 'cluster 2'])['contact count'].sum().reset_index()
            
        # Calculate the GLOBAL total interactions for Cluster 1 within each cancer type
        # First, get totals where it appears as cluster 1
        c1_sums = aggregated_df.groupby(['cancer_type', 'cluster 1'])['contact count'].sum().reset_index()
        c1_sums.columns = ['cancer_type', 'cluster 1', 'total c1 interactions']
                
        # Merge this global total back into your summary dataframe
        # This ensures every row with (cancer_type, cluster 1) has the exact same denominator
        summary_df = aggregated_df.merge(c1_sums, on=['cancer_type', 'cluster 1'])
                
        # Recompute frequency based on global interaction total
        summary_df['frequency'] = summary_df['contact count'] / summary_df['total c1 interactions'] 
           
        # High-speed vectorized mapping
        keys = list(zip(summary_df['cancer_type'], summary_df['cluster 1'], summary_df['cluster 2']))
        indices = [pair_to_idx[k] for k in keys]
        freq_matrix[indices, i] = summary_df['frequency'].values
        
    freq_cols = [f'freq_{j}' for j in range(n_permutations)]
    freq_df = pd.DataFrame(freq_matrix, columns = freq_cols) 
    null_freq_dist = pd.concat([null_freq_dist.reset_index(drop=True), freq_df], axis=1) 


Overall Permutation Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Cancer Type:   0%|          | 0/5 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/396 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/150 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/209 [00:00<?, ?it/s]

...

TCGA Slides:   0%|          | 0/396 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/150 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/209 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/398 [00:00<?, ?it/s]

TCGA Slides:   0%|          | 0/158 [00:00<?, ?it/s]

In [11]:
freq_matrix

array([[0.01801088, 0.01790179],
       [0.01468312, 0.01481076],
       [0.02183816, 0.02174418],
       ...,
       [0.0192083 , 0.01922774],
       [0.01100168, 0.01123018],
       [0.01117899, 0.01117751]], shape=(6125, 2))

In [12]:
null_freq_dist

,cancer_type,cluster 1,cluster 2,freq_0,freq_1
0,coad,0,1,0.018011,0.017902
1,coad,0,2,0.014683,0.014811
2,coad,0,3,0.021838,0.021744
3,coad,0,4,0.021108,0.020998
4,coad,0,5,0.012982,0.012894
...,...,...,...,...,...
6120,read,46,48,0.019232,0.019333
6121,read,46,49,0.011172,0.011281
6122,read,47,48,0.019208,0.019228
6123,read,47,49,0.011002,0.011230


In [31]:
null_freq_dist.to_pickle('pkl/cluster_50_permutation_test_1.pkl') 